# Improve the Model

In the previous page, we trained a baseline XGBoost regressor on the Australian rental market dataset. Our hand-picked configuration reached roughly:

- **MAE ≈ \$124 / week**
- **RMSE ≈ \$257 / week**
- **R² ≈ 0.45**

That is a useful starting point but not yet production-grade. In this page we walk through four practical levers that real pricing teams use to lift model quality:

1. **Target transformation**: handling the right-skewed rent distribution with a log transform.
2. **Feature engineering**: extracting signal from `amenities`, `description`, and geographic data.
3. **Hyperparameter tuning**: running a randomized search instead of guessing one configuration.
4. **Integrating additional data**: ideas for enriching the dataset with external sources.

:::{important} Goal
We first measure each technique **on its own** against the baseline, keeping the model settings fixed (an *ablation*). Then we combine them and tune the hyperparameters. The point is not to chase a perfect score, but to show _how_ an analyst iterates on a model and checks which changes actually help.
:::

In [1]:
import numpy as np
import pandas as pd

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder

import xgboost
from xgboost import XGBRegressor

import plotly.express as px
import plotly.io as pio

from scipy.stats import randint, uniform

pio.templates.default = "simple_white"

print(f"pandas version:       {pd.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"xgboost version:      {xgboost.__version__}")

pandas version:       3.1.1
scikit-learn version: 1.9.1
xgboost version:      3.4.1


## Load Data


In [2]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/bdi593/datasets/main/australian-rental-market-2026/australian_rental_market_2026.csv"
)
df.shape

(6767, 16)

We will hold out the same 20% test set we used in the baseline notebook so that the comparison is apples-to-apples.


In [3]:
target = "price_display"
y = df[target]

df_train, df_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42
)

print("Train:", df_train.shape, "Test:", df_test.shape)


Train: (5413, 16) Test: (1354, 16)


## 1. Target Transformation

The histogram in the baseline notebook showed that `price_display` is strongly right-skewed: a small number of luxury listings stretch into the \$3,000–\$5,000+ per week range. When the target is skewed, squared-error loss becomes dominated by these extreme observations and the model effectively over-fits to a handful of rows.

A standard remedy is to model `log(1 + price)` instead of `price`. Predictions are then exponentiated back to dollars before evaluation.

:::{tip} Why `log1p` and not `log`?
`np.log1p(x)` computes `log(1 + x)`, which is well-defined when `x = 0` and is numerically more stable than `log(x)` for small positive values. The inverse is `np.expm1(y) = exp(y) - 1`.
:::


In [4]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

fig = px.histogram(
    x=y_train_log,
    nbins=50,
    title="log(1 + weekly rent), much closer to symmetric",
    template="simple_white",
)
fig.update_layout(xaxis_title="log1p(price_display)", yaxis_title="count")
fig.show()


## 2. Feature Engineering

The baseline only used the structured columns. Several potentially informative fields were ignored. We will add three groups of features:

1. **Amenity flags** parsed from the comma-separated `amenities` text.
2. **Description length and TF–IDF features** from the `description` column.
3. **Geographic features**: rough distances to the five largest Australian capital cities, and a suburb-level rent encoding.

### 2a. Amenity flags

The `amenities` column is a free-text list such as `"Air Conditioning, Balcony, Dishwasher"`. Splitting it gives us a small set of boolean features that are easy to interpret.


In [5]:
# Each keyword is matched as a substring of the lowercased amenities text.
# Check keywords against the actual values: this data says "Built-in Wardrobes",
# not "built in robes".
AMENITY_KEYWORDS = [
    "air conditioning",
    "balcony",
    "dishwasher",
    "pool",
    "gym",
    "garage",
    "furnished",
    "heating",
    "built-in wardrobes",
    "study",
]


def amenity_col(keyword):
    return "amen_" + keyword.replace(" ", "_").replace("-", "_")


def add_amenity_flags(frame: pd.DataFrame) -> pd.DataFrame:
    text = frame["amenities"].fillna("").str.lower()
    out = frame.copy()
    for kw in AMENITY_KEYWORDS:
        out[amenity_col(kw)] = text.str.contains(kw, regex=False).astype(int)
    return out


df_train_fe = add_amenity_flags(df_train)
df_test_fe = add_amenity_flags(df_test)

amenity_cols = [amenity_col(kw) for kw in AMENITY_KEYWORDS]
df_train_fe[amenity_cols].mean().round(3).to_frame("share_of_listings")

,share_of_listings
amen_air_conditioning,0.408
amen_balcony,0.143
amen_dishwasher,0.255
amen_pool,0.045
amen_gym,0.028
amen_garage,0.563
amen_furnished,0.052
amen_heating,0.128
amen_built_in_wardrobes,0.399
amen_study,0.044


:::{tip} Check every new feature

A flag that is 0 for every listing adds nothing. An earlier version of this notebook searched for "built in robes", which never appears in this data, so that flag was silently all zeros. Printing the share of listings with each flag, as above, catches this kind of bug.

:::

### 2b. Description-based features

Listings with longer, richer descriptions may be premium properties (more to brag about) or carefully marketed rentals. We add the raw description length as a feature and a small set of TF–IDF features that capture words like _luxury_, _renovated_, _waterfront_, _new_, etc.

:::{seealso} Why limit `max_features`?
TF–IDF on tens of thousands of unique tokens would explode the feature space and hurt XGBoost's training speed. Capping `max_features` at a few hundred keeps the model nimble while preserving the highest-information words.
:::

In [6]:
df_train_fe["desc_length"] = df_train_fe["description"].fillna("").str.len()
df_test_fe["desc_length"] = df_test_fe["description"].fillna("").str.len()

tfidf = TfidfVectorizer(
    max_features=200,
    stop_words="english",
    ngram_range=(1, 1),
    min_df=20,
)
X_text_train = tfidf.fit_transform(df_train_fe["description"].fillna(""))
X_text_test = tfidf.transform(df_test_fe["description"].fillna(""))

tfidf_cols = [f"tfidf_{w}" for w in tfidf.get_feature_names_out()]
X_text_train_df = pd.DataFrame(
    X_text_train.toarray(), columns=tfidf_cols, index=df_train_fe.index
)
X_text_test_df = pd.DataFrame(
    X_text_test.toarray(), columns=tfidf_cols, index=df_test_fe.index
)

print(f"TF-IDF feature count: {len(tfidf_cols)}")


TF-IDF feature count: 200


### 2c. Geographic feature engineering

Latitude and longitude are useful, but trees split on them axis-by-axis. Two additions may help:

1. **Distance to the nearest major city.** A rough Haversine distance to each of Sydney, Melbourne, Brisbane, Perth, and Adelaide.
2. **A suburb-level rent summary.** The typical rent of other listings in the same suburb can be a useful prior. Turning a category (the suburb) into a number based on the target is called **target encoding**.

:::{caution} Target encoding can leak in two ways

**Test-set leakage.** The suburb summary must be computed from training listings only and then applied to the test set. Test-set rents must never influence a feature.

**Training-set leakage.** This one is easy to miss. If we compute each suburb's median rent on the full training set and attach it to the training rows, every row's feature includes *its own* rent. For a suburb with a single listing, the feature *is* the target. The model learns to trust the feature far more than it should, and that trust does not carry over to new listings. The code below measures how big this problem is.

:::

In [7]:
CAPITALS = {
    "sydney": (-33.8688, 151.2093),
    "melbourne": (-37.8136, 144.9631),
    "brisbane": (-27.4698, 153.0251),
    "perth": (-31.9523, 115.8613),
    "adelaide": (-34.9285, 138.6007),
}


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


def add_geo_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    for city, (clat, clon) in CAPITALS.items():
        out[f"dist_km_{city}"] = haversine_km(
            out["latitude"], out["longitude"], clat, clon
        )
    out["dist_km_nearest_capital"] = out[[f"dist_km_{c}" for c in CAPITALS]].min(axis=1)
    return out


df_train_fe = add_geo_features(df_train_fe)
df_test_fe = add_geo_features(df_test_fe)


In [8]:
# Why a plain suburb median leaks on the training set
naive_median = df_train_fe.groupby("suburb")["price_display"].transform("median")
listings_in_suburb = df_train_fe.groupby("suburb")["price_display"].transform("size")

print(
    "Training rows that are the only listing in their suburb:",
    f"{(listings_in_suburb == 1).mean():.1%}",
)
print(
    "Training rows whose suburb median equals their own rent:",
    f"{(naive_median == df_train_fe['price_display']).mean():.1%}",
)

# Keep the leaky version only to compare against it below. Do not use it.
df_train_fe["suburb_median_leaky"] = naive_median
df_test_fe["suburb_median_leaky"] = (
    df_test_fe["suburb"]
    .map(df_train_fe.groupby("suburb")["price_display"].median())
    .fillna(df_train_fe["price_display"].median())
)

Training rows that are the only listing in their suburb: 15.5%
Training rows whose suburb median equals their own rent: 26.5%


About a quarter of the training rows would see their own rent as the feature value, and about 15% are the only listing in their suburb, where the feature *is* the target.

The fix is **out-of-fold target encoding**. scikit-learn's `TargetEncoder` does this for us:

- When it is **fitted** on training data, it splits the rows into 5 folds and encodes each fold using only the other 4 folds. No row ever sees its own target.
- It also **smooths** each suburb's value toward the overall mean, so a suburb with one or two listings gets a value close to the overall mean, not an extreme one.
- When it **transforms** new data (a validation fold or the test set), it uses an encoding learned from all the rows it was fitted on.

It encodes the *mean* of the target, not the median. Because we will fit on the log of rent, the mean of log rent per suburb plays the role of a typical rent that is not pulled up by a few luxury listings.

We put the encoder inside a scikit-learn `Pipeline` together with the model. Then every call to `fit`, including each fold of cross-validation during the hyperparameter search below, refits the encoder on that fit's training rows only.

### Assemble the engineered feature matrix

We collect the new features into groups so that we can test each group on its own below. Two details:

- The categorical columns use the categories seen in the training set for both sets. A category that appears only in the test set (for example, a postcode with no training listings) becomes a missing value, which XGBoost handles natively.
- `suburb_target_enc` starts as a copy of `suburb`. The target encoder in the pipeline replaces it with a number.

In [9]:
df_train_fe["desc_length"] = df_train_fe["description"].fillna("").str.len()
df_test_fe["desc_length"] = df_test_fe["description"].fillna("").str.len()

base_numeric = ["bedrooms", "bathrooms", "parking_spaces", "latitude", "longitude"]
categorical_features = [
    "propertyType",
    "locality",
    "postcode",
    "state",
    "suburb",
    "agency_name",
]
distance_features = ["dist_km_nearest_capital"] + [f"dist_km_{c}" for c in CAPITALS]
description_features = ["desc_length"] + tfidf_cols

# Use the training categories for both sets; test-only categories become missing
for col in categorical_features:
    categories = pd.Index(df_train_fe[col].unique()).sort_values()
    dtype = pd.CategoricalDtype(categories)
    df_train_fe[col] = df_train_fe[col].astype(dtype)
    df_test_fe[col] = df_test_fe[col].where(df_test_fe[col].isin(categories)).astype(dtype)

# A copy of suburb for the target encoder to turn into a number
df_train_fe["suburb_target_enc"] = df_train_fe["suburb"]
df_test_fe["suburb_target_enc"] = df_test_fe["suburb"]

frame_cols = (
    base_numeric
    + categorical_features
    + amenity_cols
    + distance_features
    + ["desc_length", "suburb_target_enc", "suburb_median_leaky"]
)
X_train = pd.concat([df_train_fe[frame_cols], X_text_train_df], axis=1)
X_test = pd.concat([df_test_fe[frame_cols], X_text_test_df], axis=1)

base_cols = base_numeric + categorical_features
all_cols = (
    base_cols
    + amenity_cols
    + description_features
    + distance_features
    + ["suburb_target_enc"]
)
print("Baseline feature count:  ", len(base_cols))
print("Engineered feature count:", len(all_cols))

Baseline feature count:   11
Engineered feature count: 229


## Measure Each Technique on Its Own

Before combining everything, we measure each technique separately. Every row below uses the same hand-picked hyperparameters as the previous notebook and the same train/test split. Each row adds exactly **one** change to the baseline, so the difference from the baseline is the effect of that change alone. This is called an **ablation**.

The `evaluate` function builds a pipeline (adding the suburb target encoder only when that feature is used), fits it on the training set, and scores it on the test set in dollars. When the model is fitted on the log of rent, its predictions are converted back with `np.expm1` before scoring.

In [10]:
HAND_PICKED = dict(
    n_estimators=1200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    n_jobs=-1,
)


def suburb_encoder():
    """Out-of-fold, smoothed target encoding of suburb_target_enc."""
    return ColumnTransformer(
        [
            (
                "suburb_te",
                TargetEncoder(
                    target_type="continuous",
                    cv=KFold(n_splits=5, shuffle=True, random_state=42),
                ),
                ["suburb_target_enc"],
            )
        ],
        remainder="passthrough",
        verbose_feature_names_out=False,
    )


def make_pipeline(columns, model):
    steps = [("encode", suburb_encoder())] if "suburb_target_enc" in columns else []
    steps.append(("model", model))
    return Pipeline(steps).set_output(transform="pandas")


results = []


def record(name, y_pred):
    results.append(
        {
            "model": name,
            "MAE": mean_absolute_error(y_test, y_pred),
            "RMSE": mean_squared_error(y_test, y_pred) ** 0.5,
            "R2": r2_score(y_test, y_pred),
        }
    )


def evaluate(name, columns, log_target=False):
    """Fit the hand-picked model on the training set, score it on the test set."""
    pipe = make_pipeline(columns, XGBRegressor(**HAND_PICKED))
    pipe.fit(X_train[columns], np.log1p(y_train) if log_target else y_train)
    y_pred = pipe.predict(X_test[columns])
    record(name, np.expm1(y_pred) if log_target else y_pred)
    return pipe


evaluate("Baseline (hand-picked, 11 features)", base_cols)
evaluate("+ log target", base_cols, log_target=True)
evaluate("+ amenity flags", base_cols + amenity_cols)
evaluate("+ description length and TF-IDF", base_cols + description_features)
evaluate("+ distances to capitals", base_cols + distance_features)
evaluate("+ suburb target encoding (out-of-fold)", base_cols + ["suburb_target_enc"])
evaluate("+ suburb median (leaky, do not use)", base_cols + ["suburb_median_leaky"])
evaluate("All of the above except the leaky median", all_cols, log_target=True)

pd.DataFrame(results).set_index("model").round(3)

,MAE,RMSE,R2
model,,,
"Baseline (hand-picked, 11 features)",124.190,257.241,0.455
+ log target,124.702,259.789,0.444
+ amenity flags,124.830,260.759,0.440
+ description length and TF-IDF,124.886,264.205,0.425
+ distances to capitals,122.722,256.532,0.458
+ suburb target encoding (out-of-fold),123.670,257.230,0.455
"+ suburb median (leaky, do not use)",131.458,265.969,0.418
All of the above except the leaky median,125.005,262.313,0.434


With the hand-picked hyperparameters, no single technique makes a large difference:

- Most rows are within about a dollar of the baseline MAE. On a test set of 1,354 listings, differences that small could easily come from the particular split.
- Distances to capital cities give the clearest gain, about \$1.50 of MAE.
- The log target, the amenity flags, and the TF-IDF features do not help on their own with these settings. The TF-IDF features raise RMSE.
- The out-of-fold suburb encoding scores about the same as the baseline. The **leaky** in-sample median is clearly worse: MAE rises by about \$7. On the training rows the leaky feature often equals the answer, so the model learns to trust it, and that trust does not carry over to new listings.
- Adding everything at once with the same settings is slightly *worse* than the baseline. These hyperparameters were picked for 11 features; with 229 features the model has many more ways to fit noise.

Features and hyperparameters interact, which is why the next step tunes them together.

## 3. Hyperparameter Tuning with `RandomizedSearchCV`

In the baseline notebook we picked one set of hyperparameters by hand. A more rigorous approach is to _search_ over a distribution of plausible values and let cross-validation on the training set pick the best combination.

We use `RandomizedSearchCV` rather than `GridSearchCV` because it gives us much more coverage of the search space for the same number of fits. We search over the full pipeline (target encoder plus model), so the encoder is refitted inside every cross-validation fold. We optimize on the **log-transformed target** so the search is robust to skew.

:::{tip} Random search vs grid search
A 4-dimensional grid with 5 values per axis is `5⁴ = 625` configurations. A random search of 30 draws from continuous distributions typically finds a near-best configuration in a tiny fraction of the time.
:::

:::{warning} Long-running code ahead

Each search below fits 30 models (10 configurations × 3 folds) plus one final refit, and we run the search twice. To keep the total run time to several minutes, each search tries only 10 random configurations; a real project would try more (30 to 100). You do not need to run it to understand the concept, but feel free to experiment with it if you have the resources.

:::

In [11]:
param_dist = {
    "model__n_estimators": randint(400, 1500),
    "model__max_depth": randint(4, 10),
    "model__learning_rate": uniform(0.02, 0.15),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
    "model__reg_lambda": uniform(0.5, 4.0),
    "model__min_child_weight": randint(1, 10),
}


def run_search(columns):
    """Cross-validated random search over the pipeline, on the log target."""
    search = RandomizedSearchCV(
        estimator=make_pipeline(
            columns,
            XGBRegressor(
                objective="reg:squarederror",
                tree_method="hist",
                enable_categorical=True,
                random_state=42,
                n_jobs=-1,
            ),
        ),
        param_distributions=param_dist,
        n_iter=10,
        scoring="neg_mean_absolute_error",
        cv=KFold(n_splits=3, shuffle=True, random_state=42),
        verbose=1,
        random_state=42,
        n_jobs=1,  # XGBoost itself is parallel; nesting can over-subscribe CPUs
    )
    search.fit(X_train[columns], np.log1p(y_train))
    return search


search = run_search(all_cols)
search.best_params_

Fitting 3 folds for each of 10 candidates, totalling 30 fits


{'model__colsample_bytree': np.float64(0.6092249700165663),
 'model__learning_rate': np.float64(0.09871619903875838),
 'model__max_depth': 5,
 'model__min_child_weight': 3,
 'model__n_estimators': 1231,
 'model__reg_lambda': np.float64(2.3670515729919197),
 'model__subsample': np.float64(0.9439761626945282)}

:::{tip} Avoiding nested parallelism

When using `RandomizedSearchCV` with models like XGBoost, it's important to avoid nested parallelism, where both the model and the search process try to use all CPU cores at the same time. XGBoost already parallelizes training internally (via `n_jobs=-1`), so setting `n_jobs=1` in `RandomizedSearchCV` ensures that only one model is trained at a time, preventing CPU oversubscription. If both levels were parallelized, multiple models would compete for the same cores, leading to slower performance due to overhead and resource contention. The general rule is to parallelize at only one level - either within the model or across the hyperparameter search - not both.

:::


## Evaluate the Improved Model

`RandomizedSearchCV` refits the best pipeline on the full training set. We predict on the held-out test set and exponentiate predictions back to dollars before computing the metrics, then add the result to the comparison table.

In [12]:
best_model = search.best_estimator_

y_pred = np.expm1(best_model.predict(X_test[all_cols]))
record("All features + log target + random search", y_pred)

df_results = pd.DataFrame(results).set_index("model").round(3)
df_results

,MAE,RMSE,R2
model,,,
"Baseline (hand-picked, 11 features)",124.190,257.241,0.455
+ log target,124.702,259.789,0.444
+ amenity flags,124.830,260.759,0.440
+ description length and TF-IDF,124.886,264.205,0.425
+ distances to capitals,122.722,256.532,0.458
+ suburb target encoding (out-of-fold),123.670,257.230,0.455
"+ suburb median (leaky, do not use)",131.458,265.969,0.418
All of the above except the leaky median,125.005,262.313,0.434
All features + log target + random search,118.142,248.271,0.493


### Is the gain from the features or from the tuning?

The tuned model changed two things at once: it uses all the engineered features *and* searched hyperparameters. To separate them, run exactly the same search (same candidates, same folds) on the 11 baseline features with the log target.

In [13]:
search_base = run_search(base_cols)
y_pred_base = np.expm1(search_base.best_estimator_.predict(X_test[base_cols]))
record("Baseline features + log target + random search", y_pred_base)

df_results = pd.DataFrame(results).set_index("model").round(3)
df_results

Fitting 3 folds for each of 10 candidates, totalling 30 fits


,MAE,RMSE,R2
model,,,
"Baseline (hand-picked, 11 features)",124.190,257.241,0.455
+ log target,124.702,259.789,0.444
+ amenity flags,124.830,260.759,0.440
+ description length and TF-IDF,124.886,264.205,0.425
+ distances to capitals,122.722,256.532,0.458
+ suburb target encoding (out-of-fold),123.670,257.230,0.455
"+ suburb median (leaky, do not use)",131.458,265.969,0.418
All of the above except the leaky median,125.005,262.313,0.434
All features + log target + random search,118.142,248.271,0.493


Tuning helps with both feature sets. On the 11 baseline features, the search lowers MAE from about \$124 to about \$120. With all the engineered features it reaches about \$118. So most of the improvement comes from tuning, and the engineered features add a further \$2 or so once the hyperparameters suit them.

### Inspect feature importance

Looking at which features the model relies on is a quick sanity check. In the chart below, `bathrooms` dominates by gain, followed by `bedrooms`, `postcode`, and the distance to the nearest capital.

The suburb target encoding does not make the top 20. Once it can no longer see each listing's own rent, it adds little that `postcode`, `locality`, and the coordinates do not already tell the model. An earlier version of this notebook used the leaky in-sample median, and there the suburb median was the top feature: a typical symptom of target leakage.

In [14]:
final_xgb = best_model.named_steps["model"]
importances = (
    pd.Series(final_xgb.feature_importances_, index=final_xgb.feature_names_in_)
    .sort_values(ascending=False)
    .head(20)
)

fig = px.bar(
    importances.iloc[::-1],
    orientation="h",
    title="Top 20 features by XGBoost gain",
    labels={"value": "Importance", "index": "Feature"},
    height=600,
    template="simple_white",
)
fig.update_layout(showlegend=False)
fig.show()

## 4. Integrating Additional Data

Even with careful feature engineering and tuning, we are still limited by what is _in the dataset_. A real pricing engine would enrich each listing with external context.

Useful sources include:

- **Census / ABS data** at the SA2 or postcode level: median household income, age distribution, household composition, employment rate.
- **School catchment quality**: ICSEA scores, NAPLAN performance.
- **Transit accessibility**: walking distance to the nearest train/tram station, commute time to the CBD via Google Distance Matrix or OSRM.
- **Points of interest**: number of cafés, supermarkets, parks within 1 km.
- **Macro indicators**: RBA cash rate, CoreLogic price index, vacancy rates.
- **Listing dynamics**: days on market, view counts, number of price drops.

### Explanations of the above features:

**SA2 (Statistical Area Level 2)**\
A geographic unit defined by the Australian Bureau of Statistics (ABS). It represents a medium-sized area (typically a suburb or group of suburbs) designed to reflect communities with similar social and economic characteristics. Using SA2-level data lets you attach neighborhood-level features like median income or employment rates to each listing.

**ABS (Australian Bureau of Statistics)**

Australia's official statistics agency. It provides census data and other demographic/economic indicators (income, age distribution, etc.), often aggregated at levels like SA2 or postcode.

**ICSEA (Index of Community Socio-Educational Advantage)**

A score assigned to schools that reflects the socio-educational background of their students (based on factors like parental income, education, and occupation). Higher ICSEA generally indicates a more advantaged student population, which is often correlated with perceived school quality and can influence property prices.

**NAPLAN (National Assessment Program - Literacy and Numeracy)**

A standardized test taken by Australian students in Years 3, 5, 7, and 9. School-level NAPLAN results are commonly used as a proxy for academic performance, and thus another signal of school quality for nearby properties.

**RBA Cash Rate**

The benchmark interest rate set by the Reserve Bank of Australia (RBA). It influences mortgage rates and borrowing costs, making it a key macroeconomic driver of housing demand and prices.

**CoreLogic Price Index**

A widely used property market index that tracks housing price trends over time across regions. It provides a sense of broader market movement (e.g., whether prices are rising or falling overall).

**Vacancy Rates**

The percentage of rental properties that are unoccupied in a given area. Lower vacancy rates typically indicate strong rental demand, which can push prices up.

**Transit and POI metrics (not acronyms but important)**

- _Transit accessibility_: Measures like distance to the nearest station or commute time to the city center (CBD), often computed via APIs (e.g., Google Distance Matrix).
- _Points of interest (POIs)_: Counts of nearby amenities (cafés, parks, supermarkets), which help quantify neighborhood livability.

:::{hint} Joining external data
Most of these sources are available at the postcode or SA2 level. Once you have a lookup table keyed by postcode you can simply join it onto our DataFrame. Unlike a target encoding, external data does not use our own rents, so it cannot leak the target.
:::

Recall that this is an Australian rental market dataset, so the external data would be Australian-specific, but the general principle applies everywhere: the more you can tell about a property and its surroundings, the better the model can learn patterns in how those factors influence price.


## Summary

All rows below are scored on the same held-out test set (copied from the tables above).

| Iteration                                            | Test MAE (AUD/week) | R²    |
| ---------------------------------------------------- | ------------------- | ----- |
| Baseline: hand-picked settings, 11 features          | 124.2               | 0.455 |
| + log target only                                    | 124.7               | 0.444 |
| + amenity flags only                                 | 124.8               | 0.440 |
| + description length and TF-IDF only                 | 124.9               | 0.425 |
| + distances to capitals only                         | 122.7               | 0.458 |
| + suburb target encoding only (out-of-fold)          | 123.7               | 0.455 |
| + suburb median only (leaky, do not use)             | 131.5               | 0.418 |
| All features + log target, hand-picked settings      | 125.0               | 0.434 |
| Baseline features + log target + random search       | 120.5               | 0.474 |
| All features + log target + random search            | 118.1               | 0.493 |

What we learned:

- **Measure each change.** Most techniques that "usually help" did little on their own here. Only an ablation against a fixed baseline shows that.
- **Tuning mattered most in this run.** A small cross-validated search lowered MAE by about \$4 on the baseline features. The engineered features helped only once the hyperparameters were tuned for them.
- **Target encoding must be out-of-fold.** The leaky suburb median made the model worse on new listings, even though an earlier version of this notebook ranked it as the most important feature.
- **The overall gain is modest:** about \$6 per week of MAE (about 5%), with R² rising from about 0.46 to about 0.49. With one test split and a 10-candidate search, expect these numbers to move a little if you change the seed or the search budget.